# mBERT

This notebook trains a **two-stage hierarchical mBERT model** for multilingual (Bengali + Nepali) hate speech detection.
- **Stage 1 (Binary):** Hate vs. No Hate  
- **Stage 2 (Multi-label):** Insult / Violence / Sexual / Racism / Religious

In [1]:
TEXT_COL     = "Comment"
BINARY_LABEL = "Hate/NoHate"
SUBTYPE_COLS = ["Insult", "Violence", "Sexual", "Racism", "Religious"]
LABEL_COLS   = [BINARY_LABEL] + SUBTYPE_COLS

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128   # mBERT max, we had used 300 in TF-IDF for SVM but we will stick with 128 for BERT
SEED       = 42
OUTPUT_DIR = "mbert_hierarchical_hatespeech"

print("Config loaded.")
print("Label columns:", LABEL_COLS)

Config loaded.
Label columns: ['Hate/NoHate', 'Insult', 'Violence', 'Sexual', 'Racism', 'Religious']


In [ ]:
import importlib, json, os, random, re, subprocess, sys, unicodedata
from pathlib import Path

def ensure_packages(packages):
    missing = []
    for import_name, package_name in packages:
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(package_name)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_packages([
    ("pandas",      "pandas"),
    ("numpy",       "numpy"),
    ("sklearn",     "scikit-learn"),
    ("torch",       "torch"),
    ("datasets",    "datasets"),
    ("transformers","transformers"),
    ("evaluate",    "evaluate"),
    ("iterstrat",   "iterative-stratification"),
    ("openpyxl",    "openpyxl"),
    ("matplotlib",  "matplotlib"),
    ("seaborn",     "seaborn"),
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import Dataset as HfDataset, DatasetDict
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, f1_score, hamming_loss,
    jaccard_score, multilabel_confusion_matrix,
    precision_recall_fscore_support, roc_curve
)
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback,
    Trainer, TrainingArguments,
)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", DEVICE)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [ ]:
#1 
# Load dataset

DATA_PATH = "../Data/HateSpeechData.xlsx"
df = pd.read_excel(DATA_PATH)

print(f"Using dataset: {DATA_PATH}")
print("Raw shape:", df.shape)
df.head()
print("\nColumns:", list(df.columns))

In [ ]:
#2
#  Standardize column names (strip whitespace)
df.columns = df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

missing_cols = [c for c in [TEXT_COL] + LABEL_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}. Found: {list(df.columns)}")

# Ensure labels are integer 0/1
for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Ensure text is string
df[TEXT_COL] = df[TEXT_COL].astype(str)

print("Columns OK:", [TEXT_COL] + LABEL_COLS)
print("Null text rows:", df[TEXT_COL].isna().sum())
print("\nLabel value counts:")
for c in LABEL_COLS:
    print(f"  {c}: {df[c].value_counts().to_dict()}")


In [ ]:
# Rows where a subtype is flagged but binary hate label says No Hate
mask = (df[SUBTYPE_COLS].sum(axis=1) > 0) & (df[BINARY_LABEL] == 0)
print(f"Inconsistent rows found: {mask.sum()} — correcting Hate/NoHate to 1")
df.loc[mask, BINARY_LABEL] = 1

# Verify fix
remaining = ((df[SUBTYPE_COLS].sum(axis=1) > 0) & (df[BINARY_LABEL] == 0)).sum()
print(f"Inconsistent rows remaining after fix: {remaining}")


Here, 

- The SVM used `clean_text()` which **strips all punctuation** with a character whitelist. That's correct for TF-IDF but wrong for mBERT, we know that mBERT was pretrained on natural text that includes `!`, `?`, `"`, `,` etc., and those tokens carry meaning in its embeddings.
- We will, however, **collapse** repeated punctuation (`!!!` → `!`) rather than removing it entirely.
- We do **not** pre-truncate to 300 words — the tokenizer handles truncation at 128 tokens.

In [ ]:
# ── Regex patterns
URL_PATTERN            = re.compile(r"https?://\S+|www\.\S+")
USER_PATTERN           = re.compile(r"@\w+")
HASHTAG_PATTERN        = re.compile(r"#(\w+)")
CONTROL_SPACE_PATTERN  = re.compile(r"[\r\n\t]+")
MULTISPACE_PATTERN     = re.compile(r"\s+")
REPEATED_PUNCT_PATTERN = re.compile(r"([!?.,])\1+")
NEPALI_PUNCT_PATTERN   = re.compile(r"[।॥|]+")
ELLIPSIS_PATTERN       = re.compile(r"\.{2,}")
CHAR_REPEAT_PATTERN    = re.compile(r"(.)\1{2,}")

# Extended emoji block (matches SVM's more thorough coverage)
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002600-\U000026FF"
    "\U00002700-\U000027BF"
    "]+", flags=re.UNICODE
)

def normalize_text(text: str) -> str:
    """
    DL-safe cleaning for mBERT. Key differences from SVM clean_text():
      - Does NOT strip punctuation (mBERT needs it)
      - Does NOT pre-truncate (tokenizer handles MAX_LENGTH=128)
      - DOES remove emojis, normalize Unicode, replace social tokens
    """
    text = unicodedata.normalize("NFKC", str(text))

    # Remove zero-width / BOM characters
    text = text.replace("\u200b", " ").replace("\ufeff", " ")

    # Collapse control whitespace
    text = CONTROL_SPACE_PATTERN.sub(" ", text)

    # Replace social tokens with placeholders mBERT understands
    text = URL_PATTERN.sub(" [URL] ", text)
    text = USER_PATTERN.sub(" [USER] ", text)
    text = HASHTAG_PATTERN.sub(r" \1 ", text)   # unwrap hashtag, keep word

    # Remove emojis (they tokenize to [UNK] in mBERT — noise, not signal)
    text = EMOJI_PATTERN.sub(" ", text)

    # Remove Nepali/Bengali sentence-end punctuation (not meaningful to mBERT)
    text = NEPALI_PUNCT_PATTERN.sub(" ", text)

    # Remove ellipsis ("...") — not a meaningful mBERT token
    text = ELLIPSIS_PATTERN.sub(" ", text)

    # Collapse spam repetition ("hahahaha" → "ha", "!!!" → "!")
    text = CHAR_REPEAT_PATTERN.sub(r"\1", text)
    text = REPEATED_PUNCT_PATTERN.sub(r"\1", text)

    # Final whitespace collapse
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

# Apply to dataframe
before_rows = len(df)
df["text"] = df[TEXT_COL].apply(normalize_text)

# Drop rows that became empty after cleaning
df = df[df["text"].str.strip().str.len() > 0].copy()

# Drop rows with fewer than 2 tokens (not useful for any model)
df = df[df["text"].str.split().str.len() >= 2].copy()

# Drop duplicates (same cleaned text + same label combination)
df = df.drop_duplicates(subset=["text"] + LABEL_COLS).reset_index(drop=True)

print(f"Rows before cleaning: {before_rows}")
print(f"Rows after cleaning:  {len(df)}  (removed {before_rows - len(df)})")
print(f"\nSample before → after:")
display(df[[TEXT_COL, "text"]].head(8))